In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

np.random.seed(42)

# --- Parameters ---
n_groups = 2
n_per_group = 100
n_total = n_groups * n_per_group

# Group labels
group_labels = np.repeat(np.arange(n_groups), n_per_group)

# Design matrix: z_unimodal and z_transmodal
z_unimodal   = np.random.normal(size=n_total)
z_transmodal = np.random.normal(size=n_total)

# True group-level coefficients (α_uni, α_trans, β_uni, β_trans) × 2 groups
means = [
    [0.0,  0.5,  1.0, -0.5],   # Group 0
    [1.0, -0.5, -1.0,  0.5],   # Group 1
]

# Shared 4x4 covariance for all groups
cov = np.array([
    [0.5,  0.2,  0.1,  0.2],
    [0.2,  0.5,  0.2,  0.1],
    [0.1,  0.2,  0.7,  0.3],
    [0.2,  0.1,  0.3,  0.6]
])

# Generate group-wise coefficients
coefs = np.random.multivariate_normal(mean=means[0], cov=cov, size=n_per_group)
coefs = np.vstack([coefs,
                   np.random.multivariate_normal(mean=means[1], cov=cov, size=n_per_group)])

# Extract α and β
α_uni   = coefs[:, 0]
α_trans = coefs[:, 1]
β_uni   = coefs[:, 2]
β_trans = coefs[:, 3]

# Linear predictor and probabilities
logits = α_uni + α_trans + β_uni * z_unimodal + β_trans * z_transmodal
p = 1 / (1 + np.exp(-logits))
y = np.random.binomial(1, p)

# Assemble DataFrame
df = pd.DataFrame({
    "group_idx": group_labels,
    "z_unimodal": z_unimodal,
    "z_transmodal": z_transmodal,
    "y": y
})

print(df.head())
